## Recurrent Neural Network (RNN)

The aim of this notebook is to show a full example of how to implement from scratch a RNN, first using only NumPy and then using thorcino (this framework), building the necessary component to make it a resable and composable module.

#### Disclaimer

If you have not an experience with RNNs or NN, read the [README](./README.md) it contains a brief introduction to the topic and some resources to learn more about it.

## The Learning Task

Our RNN, will be very simple, (but general enough to be used in other tasks), the learning task is to predict the next number in a sequence of equidistant ordered numbers.

In [1]:
"""Hyper parameters (README for the complete notation)"""

N = 10 # number of samples
T = 5  # length of sequences (time steps)
D = 1  # number of features of each sequence element 
O = 1  # number of output units
H = 1  # number of hidden units


In [8]:
"""Building Dataset"""

import numpy as np

def create_sequence(len: int, start: float, stop: float) -> np.ndarray:
    return np.linspace(start, stop, len)

def create_random_sequences(n: int, len:int, min_start: int=10, seed: float=777) -> np.ndarray:
    rng = np.random.default_rng(seed)
    sequences = []
    for _ in range(n):
        start = rng.integers(min_start, size=1)[0]
        stop = start * 2
        seq = create_sequence(len+1, start, stop)

        sequences.append(seq)

    return np.stack(sequences)

A = create_random_sequences(N, T)
X, Y = A[:, :A.shape[1]-1], A[:, 1:]

print('input data:')
print(X)

print('\n\ntarget data:')
print(Y)

input data:
[[ 9.  10.8 12.6 14.4 16.2]
 [ 6.   7.2  8.4  9.6 10.8]
 [ 3.   3.6  4.2  4.8  5.4]
 [ 3.   3.6  4.2  4.8  5.4]
 [ 0.   0.   0.   0.   0. ]
 [ 6.   7.2  8.4  9.6 10.8]
 [ 4.   4.8  5.6  6.4  7.2]
 [ 9.  10.8 12.6 14.4 16.2]
 [ 3.   3.6  4.2  4.8  5.4]
 [ 1.   1.2  1.4  1.6  1.8]]


target data:
[[10.8 12.6 14.4 16.2 18. ]
 [ 7.2  8.4  9.6 10.8 12. ]
 [ 3.6  4.2  4.8  5.4  6. ]
 [ 3.6  4.2  4.8  5.4  6. ]
 [ 0.   0.   0.   0.   0. ]
 [ 7.2  8.4  9.6 10.8 12. ]
 [ 4.8  5.6  6.4  7.2  8. ]
 [10.8 12.6 14.4 16.2 18. ]
 [ 3.6  4.2  4.8  5.4  6. ]
 [ 1.2  1.4  1.6  1.8  2. ]]


In [3]:
"""Building the RNN state, hidden state, weights and bias"""

from thorcino.functions import mse


SEED = 777

rng = np.random.default_rng(SEED)

h_activation = np.tanh
o_activation = np.identity
loss = mse

P = {
    'Wxh':np.random.randn(D, H), # Input weight matrix (shared across time steps)
    'Who':np.random.randn(H, O), # Output weight matrix (shared across time steps)
    'Whh':np.random.randn(H, H), # hidden-state-to-hidden-state matrix
    'Ht':np.random.randn(N, H),  # hidden state at the input at time step t
    'bh':np.random.randn(1, H),  # hidden bias
    'bo':np.random.randn(1, 0),  # output bias
}

`compute_H` function computes the hidden state of the RNN as the following equation:

$$
\mathbf{H}_t = \phi_h \left( \mathbf{X}_t \mathbf{W}_{xh} + \mathbf{H}_{t-1} \mathbf{W}_{hh} + \mathbf{b}_h \right)
$$

`compute_output` function computes the output of the RNN as the following equation:
$$
\mathbf{O}_t = \phi_o \left( \mathbf{H}_t \mathbf{W}_{ho} + \mathbf{b}_o \right)
$$

In [ ]:
"""forward utilities"""

def compute_H(X: np.ndarray, P: dict) -> np.ndarray:
    L = X@P['WWxh'] + P['Ht']@P['Whh'] + P['bh']
    return h_activation( L )

def compute_output(P: dict) -> np.ndarray:
    L = P['Ht']@P['Who'] + P['bo']
    return o_activation( L )

def forward(X: np.ndarray, P: dict) -> tuple[np.ndarray, np.ndarray]:
    cache, loss = [], 0

    for t in range(T):
        Hprev = H
        H = compute_H(X, X[t, :], P['WWxh'], P['Whh'], Hprev, P['bh'])
        out = compute_output(H, P['Who'], P['bo'])
        loss += loss(out, Y[t, :])

        cache.append((Hprev, H, out))

    return loss, cache

## Gradient computation

$$
\mathcal{L} \left( \mathbf{O}, \mathbf{Y} \right)
= \sum_{t=1}^{T} \ell_t \left( \mathbf{O}_t, \mathbf{Y}_t \right)
$$

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{W}_{ho}}
= \sum_{t=1}^{T} \frac{\partial \ell_t}{\partial \mathbf{O}_t}
  \cdot \frac{\partial \mathbf{O}_t}{\partial \phi_o}
  \cdot \frac{\partial \phi_o}{\partial \mathbf{W}_{ho}}
= \sum_{t=1}^{T} \frac{\partial \ell_t}{\partial \mathbf{O}_t}
  \cdot \frac{\partial \mathbf{O}_t}{\partial \phi_o}
  \cdot \mathbf{H}_t
$$

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{W}_{xh}}
= \sum_{t=1}^{T} \frac{\partial \ell_t}{\partial \mathbf{O}_t}
  \cdot \frac{\partial \mathbf{O}_t}{\partial \phi_o}
  \cdot \frac{\partial \phi_o}{\partial \mathbf{H}_t}
  \cdot \frac{\partial \mathbf{H}_t}{\partial \phi_h}
  \cdot \frac{\partial \phi_h}{\partial \mathbf{W}_{xh}}
= \sum_{t=1}^{T} \frac{\partial \ell_t}{\partial \mathbf{O}_t}
  \cdot \frac{\partial \mathbf{O}_t}{\partial \phi_o}
  \cdot \mathbf{W}_{ho}
  \cdot \frac{\partial \mathbf{H}_t}{\partial \phi_h}
  \cdot \frac{\partial \phi_h}{\partial \mathbf{W}_{xh}}
$$

$$
\begin{aligned}
\frac{\partial \mathcal{L}}{\partial \mathbf{W}_{hh}}
&= \sum_{t=1}^{T} \frac{\partial \ell_t}{\partial \mathbf{O}_t}
   \cdot \frac{\partial \mathbf{O}_t}{\partial \phi_o}
   \cdot \mathbf{W}_{ho}
   \sum_{k=1}^{t} \left( \mathbf{W}_{hh}^{\top} \right)^{t-k}
   \cdot \mathbf{H}_k \\[1em]
\frac{\partial \mathcal{L}}{\partial \mathbf{W}_{xh}}
&= \sum_{t=1}^{T} \frac{\partial \ell_t}{\partial \mathbf{O}_t}
   \cdot \frac{\partial \mathbf{O}_t}{\partial \phi_o}
   \cdot \mathbf{W}_{ho}
   \sum_{k=1}^{t} \left( \mathbf{W}_{hh}^{\top} \right)^{t-k}
   \cdot \mathbf{X}_k
\end{aligned}
$$

In [9]:
"""backward utilities"""

def backward(P:dict, cache: list) -> dict:
    g = {k: np.zeros_like(v) for k, v in P.items()}
    dH_next = np.zeros((N, H))

    for t in reversed(range(T)):
        Hprev, H, out = cache[t]
        dout = 2*(out - Y[t, :])
        
        g['Who'] += H.T @ dout
        g['bo'] += dout.sum(0, keepdims=True)

        dH = dout @ P['Who'].T + dH_next
        dA = dH * (1 - H**2)
        
        g['Wxh'] += X[t, :] @ dA
        g['Whh'] += Hprev.T @ dA
        g['bh'] += dA.sum(0, keepdims=True)

        dH_next = dA @ P['Whh'].T

    return g